# 03 — Test des 5 modèles de classification

**Objectif :** comparer 5 modèles pour prédire `Churn` : Logistic Regression, Decision Tree, Random Forest, SVM et Gradient Boosting.

**Ce notebook se construit étape par étape :** d'abord les 5 modèles tels quels (sections 3 et 4), puis le traitement du déséquilibre des classes, en changeant une seule chose à la fois (section 5). Enfin, on teste si la variable `cluster` améliore la prédiction (section 6). L'optimisation viendra ensuite. Les sections 1 à 5 n'utilisent que `src/` ; la section 6 utilise les fichiers produits par `02_clustering.ipynb`.

**Règles respectées**
- `Churn` n'est jamais dans `X`.
- Imputation, standardisation et encodage sont dans un `Pipeline` : ajustés uniquement sur le train.
- La comparaison se fait par **validation croisée sur le train** ; le **test** sert à l'évaluation finale.

## 1. Imports et données

In [1]:
import sys, os, warnings
sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

from src.preprocessing import prepare, load_data, train_test, TARGET, RANDOM_STATE
from src.models import build_pipeline, build_baseline, load_split, MODEL_NAMES
from src.evaluate import stratified_cv, cv_scores, cv_fold_scores, test_metrics, plot_confusions, plot_roc

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)

X_train, X_test, y_train, y_test = train_test(prepare(load_data()))   # split stratifié, avant tout fit
assert TARGET not in X_train.columns and TARGET not in X_test.columns
print("X_train :", X_train.shape, "| X_test :", X_test.shape)
print("Churn train : %.3f | test : %.3f" % (y_train.mean(), y_test.mean()))

ImportError: cannot import name 'cv_fold_scores' from 'src.evaluate' (c:\Users\ycode\Desktop\Dev_AI\S1\Telecom-Churn-ML\src\evaluate.py)

## 2. Métriques et baseline

On regarde la classe **Churn** (celle qu'on veut détecter) :

| Métrique | Question |
|---|---|
| **Precision** | Parmi les clients prédits "churn", quelle part part vraiment ? |
| **Recall** | Parmi ceux qui partent vraiment, quelle part détecte-t-on ? |
| **F1** | Compromis entre Precision et Recall |
| **ROC-AUC** | Capacité à classer les clients par risque, indépendamment du seuil |
| **Matrice de confusion** | Nombre de bonnes détections, de clients manqués et de fausses alertes |

**Baseline :** un modèle qui prédit toujours "reste". C'est le plancher à dépasser.

In [ ]:
cv = stratified_cv(n_splits=5)      # 5 plis stratifiés, sur le train
base = cv_scores(build_baseline(), X_train, y_train, cv)
pd.Series({k: round(v, 3) for k, v in base.items() if not k.endswith("_std")}, name="baseline (CV)").to_frame().T

**Lecture :** la baseline atteint ≈ 73,5 % d'accuracy sans rien apprendre, mais son Recall est de 0 : elle ne détecte aucun client qui part. L'accuracy seule est donc trompeuse ici (≈ 26,5 % de churn).

## 3. Les 5 modèles : validation croisée sur le train

Chaque modèle est un `Pipeline` (preprocessing → modèle), avec des réglages raisonnables par défaut (l'optimisation viendra plus tard).

In [ ]:
rows = []
for name in MODEL_NAMES:
    res = cv_scores(build_pipeline(name, "none"), X_train, y_train, cv)
    res["model"] = name
    rows.append(res)
cv_table = pd.DataFrame(rows).set_index("model")
cv_table[["accuracy", "precision", "recall", "f1", "roc_auc", "f1_std", "roc_auc_std"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, metric, title in zip(axes, ["recall", "f1", "roc_auc"], ["Recall (churn)", "F1 (churn)", "ROC-AUC"]):
    sns.barplot(x=cv_table.index, y=cv_table[metric], color="#4C72B0", ax=ax)
    ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(axis="x", rotation=30)
    if metric == "roc_auc": ax.set_ylim(0.75, 0.87)
plt.tight_layout(); plt.show()

## 4. Évaluation sur le test

Chaque modèle est ré-entraîné sur tout le train puis évalué **une seule fois** sur le test.

In [ ]:
fitted = {name: build_pipeline(name, "none").fit(X_train, y_train) for name in MODEL_NAMES}

rows = {name: test_metrics(pipe, X_test, y_test) for name, pipe in fitted.items()}
rows["Baseline (toujours 'reste')"] = test_metrics(build_baseline().fit(X_train, y_train), X_test, y_test)
pd.DataFrame(rows).T.round(3).sort_values("roc_auc", ascending=False)

In [ ]:
plot_confusions(fitted, X_test, y_test, ncols=3)

In [ ]:
plot_roc(fitted, X_test, y_test)

**Lecture (sans traitement du déséquilibre) :**
- Tous les modèles dépassent la baseline en accuracy (≈ 0,78 à 0,80 contre 0,735), mais ils ne détectent qu'environ **la moitié des clients qui partent** (Recall ≈ 0,47 à 0,58) avec une Precision de 0,62 à 0,68.
- **ROC-AUC** : Gradient Boosting et Logistic Regression en tête (≈ 0,84), Decision Tree et Random Forest ≈ 0,83, SVM le plus faible (≈ 0,80).
- **F1** : de 0,54 à 0,61. Les écarts entre modèles sont petits par rapport au bruit de la validation croisée (écart-type ≈ 0,02).

Le point faible commun est le **Recall d'environ 50 %** : c'est ce que la section suivante essaie de corriger.

## 5. Le déséquilibre des classes

**Le problème.** Seulement ≈ 26,5 % des clients partent. Un modèle apprend en minimisant ses erreurs *globales* : se tromper sur un client qui part ne pèse que 26,5 % du total, alors que se tromper sur un client qui reste pèse 73,5 %. Le modèle a donc intérêt à être prudent avec la classe "churn" et à la prédire rarement : c'est ce qu'on observe (Recall ≈ 50 %).

**Deux remèdes**, testés **un par un**, avec le même protocole (mêmes plis de validation croisée, mêmes modèles) :
1. `class_weight` : on change le *coût* des erreurs ;
2. SMOTE : on change les *données* d'entraînement.

Puis on compare les trois situations (rien, `class_weight`, SMOTE) et on explique pourquoi le ROC-AUC bouge peu.

### 5.1 Étape 1 : `class_weight="balanced"` (on change le coût des erreurs)

Avec `balanced`, chaque classe reçoit un poids inversement proportionnel à sa fréquence :
`poids = n_total / (2 × n_classe)`. Une erreur sur un client qui part coûte alors plusieurs fois plus cher qu'une erreur sur un client qui reste.

In [ ]:
n, n_pos = len(y_train), int(y_train.sum())
n_neg = n - n_pos
w_pos, w_neg = n / (2 * n_pos), n / (2 * n_neg)
print(f"poids churn = {w_pos:.2f} | poids reste = {w_neg:.2f} | une erreur 'churn' coûte {w_pos / w_neg:.1f}x plus")

In [ ]:
cols = ["precision", "recall", "f1", "roc_auc"]

rows = []
for name in MODEL_NAMES:
    pipe = build_pipeline(name, "class_weight")
    if pipe is None:                       # Gradient Boosting n'a pas de paramètre class_weight
        print(f"{name} : class_weight indisponible, ignoré")
        continue
    res = cv_scores(pipe, X_train, y_train, cv)
    res["model"] = name
    rows.append(res)
cv_cw = pd.DataFrame(rows).set_index("model")

print("\nclass_weight='balanced' (validation croisée)")
display(cv_cw[cols].round(3))
print("Différence par rapport à la section 3 ('none')")
display((cv_cw[cols] - cv_table.loc[cv_cw.index, cols]).round(3))

**Lecture :**
- **Logistic Regression, Decision Tree, SVM** : le Recall monte fortement (Logistic Regression 0,53 → 0,80 ; Decision Tree 0,56 → 0,78 ; SVM 0,47 → 0,78), mais la Precision chute (Logistic Regression 0,67 → 0,52). Le F1 gagne entre 3 et 7 points : on détecte beaucoup plus de clients qui partent, au prix de plus de fausses alertes.
- **Random Forest** : aucun gain (Recall 0,49 → 0,47, F1 0,555 → 0,545). Hypothèse : des arbres profonds s'ajustent presque parfaitement aux données d'entraînement, donc les poids changent peu leurs prédictions (non vérifiée ici).
- **Gradient Boosting** : pas de `class_weight` dans cette implémentation, donc non testé.
- **ROC-AUC** : quasiment identique pour Logistic Regression, Decision Tree et Random Forest. Exception : le **SVM** passe de 0,80 à 0,83. On y revient en 5.4.

### 5.2 Étape 2 : SMOTE (on change les données d'entraînement)

**SMOTE** (*Synthetic Minority Over-sampling Technique*) crée de **nouveaux clients "churn" synthétiques** : pour un client qui part, il choisit un de ses plus proches voisins parmi les clients qui partent et place un nouveau point sur le segment qui les relie. L'entraînement se fait ensuite sur des classes équilibrées.

**Règle essentielle :** SMOTE ne doit agir que sur les données d'entraînement. Il est donc placé **dans le Pipeline** : dans chaque pli de la validation croisée, il ne voit que les données d'entraînement du pli, jamais celles de validation, et il n'agit pas au moment du `predict`. Si on l'appliquait avant la validation croisée, des points synthétiques fabriqués à partir de clients de validation fuiraient dans l'entraînement.

L'illustration ci-dessous montre l'effet de SMOTE sur les classes (à titre d'exemple ; dans le Pipeline c'est automatique).

In [ ]:
pre = build_pipeline("Logistic Regression", "none").named_steps["pre"]
Z = pre.fit_transform(X_train)
Z_res, y_res = SMOTE(random_state=RANDOM_STATE).fit_resample(Z, y_train)

print("Avant SMOTE :", y_train.value_counts().to_dict(), "| après :", pd.Series(y_res).value_counts().to_dict())

In [ ]:
rows = []
for name in MODEL_NAMES:
    res = cv_scores(build_pipeline(name, "smote"), X_train, y_train, cv)
    res["model"] = name
    rows.append(res)
cv_smote = pd.DataFrame(rows).set_index("model")

print("SMOTE (validation croisée)")
display(cv_smote[cols].round(3))
print("Différence par rapport à la section 3 ('none')")
display((cv_smote[cols] - cv_table[cols]).round(3))

**Lecture :**
- **Logistic Regression, Decision Tree, SVM** : même profil que `class_weight` (Recall ≈ 0,73 à 0,79, Precision ≈ 0,51 à 0,54, F1 ≈ 0,61 à 0,63).
- **Gradient Boosting** : Recall 0,52 → 0,66, Precision 0,66 → 0,59, F1 0,58 → 0,62.
- **Random Forest** : cette fois un gain modeste (Recall 0,49 → 0,58, F1 0,555 → 0,59), contrairement à `class_weight`.
- **ROC-AUC** : peu de changement (écarts d'au plus 0,007), sauf le **SVM** (0,80 → 0,83).

### 5.3 Étape 3 : comparer les trois situations

In [ ]:
all_cv = pd.concat([cv_table.assign(strategy="none"),
                    cv_cw.assign(strategy="class_weight"),
                    cv_smote.assign(strategy="smote")]).reset_index()
order = ["none", "class_weight", "smote"]

summary = pd.concat({m: all_cv.pivot(index="model", columns="strategy", values=m)[order] for m in cols}, axis=1)
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 3.8))
for ax, metric, title in zip(axes, cols, ["Precision", "Recall", "F1", "ROC-AUC"]):
    sns.barplot(data=all_cv, x="model", y=metric, hue="strategy", hue_order=order, ax=ax,
                palette={"none": "#9AA5B1", "class_weight": "#4C72B0", "smote": "#C44E52"})
    ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(axis="x", rotation=30)
    if metric == "roc_auc": ax.set_ylim(0.75, 0.87)
    if metric != "precision": ax.get_legend().remove()
plt.tight_layout(); plt.show()

**Lecture :**
- Le rééquilibrage déplace les modèles vers **plus de Recall et moins de Precision**, avec un meilleur F1 pour la plupart (sauf Random Forest avec `class_weight`).
- **`class_weight` et SMOTE donnent des résultats très proches** pour Logistic Regression, Decision Tree et SVM (écart de F1 ≤ 0,003, bien en dessous du bruit ≈ 0,02). `class_weight` est plus simple et plus rapide ; SMOTE est le seul remède disponible pour Gradient Boosting.
- Aucune stratégie n'est "la meilleure" en absolu : le choix dépend du compromis voulu entre clients détectés (Recall) et fausses alertes (Precision).

### 5.4 Pourquoi le ROC-AUC change peu ?

Un modèle produit pour chaque client un **score** (une probabilité de churn). La décision "il part" est prise quand ce score dépasse un **seuil** (0,5 par défaut).

- **Precision, Recall, F1** dépendent du seuil : le baisser fait prédire "churn" plus souvent.
- **ROC-AUC** ne dépend pas du seuil : il mesure seulement si les clients qui partent reçoivent des scores plus élevés que ceux qui restent (la qualité du *classement*).

Pour la Logistic Regression, rééquilibrer revient surtout à **relever les scores de tous les clients** : le classement reste presque le même (ROC-AUC stable), mais plus de clients dépassent 0,5. On le vérifie : on garde la Logistic Regression **sans traitement** et on baisse simplement le seuil.

In [ ]:
lr_none = build_pipeline("Logistic Regression", "none")
proba = cross_val_predict(lr_none, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
print("ROC-AUC (indépendant du seuil) :", round(roc_auc_score(y_train, proba), 3))

rows = []
for th in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25]:
    pred = (proba >= th).astype(int)
    rows.append({"seuil": th, "precision": precision_score(y_train, pred), "recall": recall_score(y_train, pred),
                 "f1": f1_score(y_train, pred), "% prédits churn": pred.mean() * 100})
thr = pd.DataFrame(rows).set_index("seuil")
display(thr.round(3))

# Référence : la même régression logistique avec class_weight, seuil 0,5
lr_cw = build_pipeline("Logistic Regression", "class_weight")
proba_cw = cross_val_predict(lr_cw, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
pred_cw = (proba_cw >= 0.5).astype(int)
print("class_weight, seuil 0.5 -> precision %.3f | recall %.3f | f1 %.3f | %% prédits churn %.1f | ROC-AUC %.3f" % (
    precision_score(y_train, pred_cw), recall_score(y_train, pred_cw), f1_score(y_train, pred_cw),
    pred_cw.mean() * 100, roc_auc_score(y_train, proba_cw)))

**Lecture :**
- Sans aucun traitement, en abaissant le seuil de 0,5 à environ 0,25–0,30, la Logistic Regression atteint un Recall de 0,76 à 0,81 avec une Precision de 0,51 à 0,54 et un F1 ≈ 0,63. C'est **pratiquement le résultat de `class_weight` à seuil 0,5** (Recall 0,80, Precision 0,52, F1 0,63), avec le **même ROC-AUC (≈ 0,846)**.
- Pour ce modèle, le rééquilibrage a donc surtout **déplacé la frontière de décision** (la part de clients prédits "churn" passe d'environ 21 % à 41 %) sans améliorer la capacité à classer les clients.
- **Ce n'est pas vrai pour tous les modèles** : pour le SVM, `class_weight` modifie l'entraînement lui-même (le coût des erreurs change la frontière apprise) et le ROC-AUC monte de 0,80 à 0,83. Pour le Random Forest, le Recall n'a pas bougé avec `class_weight`. C'est pourquoi on vérifie par modèle plutôt que de généraliser.
- **Conséquence pratique :** le seuil est un **levier métier** qu'on peut régler à tout moment, sans ré-entraîner : prédire "churn" plus souvent coûte des fausses alertes mais évite de perdre des clients.

### 5.5 Choix de la stratégie et retour au test

Pour chaque modèle, on retient la stratégie qui donne le meilleur **F1 en validation croisée** (le test n'intervient pas dans ce choix), puis on compare au test avec et sans rééquilibrage.

In [ ]:
best_strategy = (all_cv.loc[all_cv.groupby("model")["f1"].idxmax()]
                 .set_index("model")["strategy"].to_dict())
print("Stratégie retenue par modèle :", best_strategy)

fitted_best = {f"{n} ({best_strategy[n]})": build_pipeline(n, best_strategy[n]).fit(X_train, y_train)
               for n in MODEL_NAMES}

rows = {}
for n in MODEL_NAMES:
    rows[(n, "none")] = test_metrics(fitted[n], X_test, y_test)
    rows[(n, best_strategy[n])] = test_metrics(fitted_best[f"{n} ({best_strategy[n]})"], X_test, y_test)
pd.DataFrame(rows).T[["precision", "recall", "f1", "roc_auc"]].round(3)

In [ ]:
plot_confusions(fitted_best, X_test, y_test, ncols=3)

**Lecture (test) :**
- Le rééquilibrage améliore le **F1 de tous les modèles** (de +0,02 pour Random Forest à +0,08 pour le SVM) et fait passer le Recall de Logistic Regression, Decision Tree et SVM de 0,47–0,58 à **≈ 0,79**.
- Le prix est la Precision : elle baisse de 10 à 15 points pour ces trois modèles (≈ 0,51 à 0,53). Concrètement, pour chaque client qui part et qui est repéré, il y a environ une fausse alerte.
- Les matrices de confusion montrent le même mouvement : moins de clients qui partent manqués (case en bas à gauche), plus de clients qui restent signalés à tort (case en haut à droite).
- Le **ROC-AUC reste stable** (écarts ≤ 0,003), sauf le SVM (0,80 → 0,83). Il ne s'agit donc pas de modèles "meilleurs" pour classer les clients, mais de modèles réglés pour détecter davantage.
- Les écarts entre modèles restent petits par rapport au bruit : Logistic Regression et Gradient Boosting conservent le meilleur ROC-AUC (≈ 0,84).

## 6. Le cluster améliore-t-il la prédiction du churn ?

**La question.** Le clustering (notebook 02) a regroupé les clients en 4 segments, dont les taux de churn sont très différents (de 7 % à 57 %). On peut donc espérer que savoir "à quel segment appartient le client" aide à prédire s'il part.

**Mais** le cluster a été calculé à partir des *mêmes* variables que celles que le modèle voit déjà, et sans jamais utiliser `Churn`. L'information est-elle vraiment nouvelle ? On le mesure.

**Méthode (une seule chose change à la fois) :**

| Config | Variables | Ce qu'on compare |
|---|---|---|
| **A** | variables initiales (sans Feature Engineering, sans cluster) | référence |
| **B** | A + Feature Engineering (`n_services`, `avg_charge`, `tenure_group`) | A vs B : apport du FE |
| **C** | A + `cluster` | A vs C : apport du cluster (demandé par le sujet) |
| **D** | B + `cluster` | B vs D : cluster en plus du FE |

Tous les modèles sont entraînés avec **SMOTE** (seule stratégie disponible pour les 5 modèles). La validation croisée est **5 plis × 2 répétitions = 10 plis**, identiques d'une configuration à l'autre : on compare donc **pli par pli** (comparaison appariée). Pour chaque modèle on regarde la différence moyenne, son écart-type, et le nombre de plis (sur 10) où la nouvelle configuration est meilleure. **S'il n'y a aucun effet, on s'attend à environ 5 sur 10** (comme à pile ou face).

Attention : les plis partagent des données d'entraînement, ils ne sont pas totalement indépendants. Le décompte des plis est un indicateur, pas un test statistique.

### 6.1 Ajouter la colonne `cluster`

In [ ]:
PROCESSED = "../data/processed"
for f in ["train_with_clusters.csv", "test_with_clusters.csv"]:
    assert os.path.exists(f"{PROCESSED}/{f}"), f"{f} manquant : lancer d'abord 02_clustering.ipynb (Run All)"

Xc_train, Xc_test, yc_train, yc_test = load_split(PROCESSED)

# Vérification : mêmes clients, dans le même ordre que X_train / X_test de ce notebook
assert np.array_equal(Xc_train["tenure"].values, X_train["tenure"].values)
assert np.allclose(Xc_train["MonthlyCharges"].values, X_train["MonthlyCharges"].values)
assert np.array_equal(yc_train.values, y_train.values) and np.array_equal(yc_test.values, y_test.values)

X_train_c = X_train.assign(cluster=Xc_train["cluster"].values)
X_test_c = X_test.assign(cluster=Xc_test["cluster"].values)

prof = pd.DataFrame({
    "clients": X_train_c["cluster"].value_counts().sort_index(),
    "% clients": (X_train_c["cluster"].value_counts(normalize=True).sort_index() * 100),
    "taux de churn %": y_train.groupby(X_train_c["cluster"].values).mean() * 100,
}).round(1)
display(prof)

n_sans = build_pipeline("Logistic Regression", "none", use_cluster=False).named_steps["pre"].fit_transform(X_train_c).shape[1]
n_avec = build_pipeline("Logistic Regression", "none", use_cluster=True).named_steps["pre"].fit_transform(X_train_c).shape[1]
print(f"Colonnes après preprocessing : {n_sans} sans cluster -> {n_avec} avec cluster (one-hot : {n_avec - n_sans} colonnes en plus)")

Le `cluster` (0 à 3) est une **catégorie**, pas une quantité : on l'encode en one-hot (4 colonnes en plus), sinon le modèle croirait que le cluster 3 est "plus grand" que le cluster 0. Les noms des segments sont dans le notebook 02. Les taux de churn par cluster confirment l'écart (de quelques % à plus de 50 %).

### 6.2 Préparation des comparaisons

In [ ]:
configs = {"A": (False, False), "B": (False, True), "C": (True, False), "D": (True, True)}   # (use_cluster, use_fe)
cv_rep = stratified_cv(n_splits=5, n_repeats=2)      # 10 plis, identiques pour toutes les configurations
fold = {}                                             # scores pli par pli : fold[(config, modèle)]

def run(cfg):
    use_cluster, use_fe = configs[cfg]
    for name in MODEL_NAMES:
        fold[(cfg, name)] = cv_fold_scores(build_pipeline(name, "smote", use_cluster, use_fe), X_train_c, y_train, cv_rep)

def compare(new, ref, metric):
    rows = []
    for name in MODEL_NAMES:
        d = fold[(new, name)][metric] - fold[(ref, name)][metric]      # différence pli par pli
        rows.append({"model": name, ref: fold[(ref, name)][metric].mean(), new: fold[(new, name)][metric].mean(),
                     "diff (new-ref)": d.mean(), "écart-type diff": d.std(), "plis gagnés /10": int((d > 0).sum())})
    return pd.DataFrame(rows).set_index("model")

def report(new, ref):
    for metric in ["roc_auc", "f1"]:
        out = compare(new, ref, metric)
        print(f"--- {metric} : {new} vs {ref} ---")
        display(out.round(4))
        print(f"différence moyenne : {out['diff (new-ref)'].mean():+.4f} | plis gagnés au total : {out['plis gagnés /10'].sum()} / {10 * len(MODEL_NAMES)}\n")

### 6.3 Étape 1 : A vs C (le cluster seul)

Chaque étape de cette section prend environ une minute (le total de la section est d'environ 3 minutes).

In [ ]:
run("A")
run("C")
report("C", "A")

**Lecture :**
- **ROC-AUC** : les différences sont minuscules (au plus 0,0025 en valeur absolue, moyenne −0,0006) et le total est de **24 plis gagnés sur 50**, soit exactement ce qu'on attend d'un pile ou face.
- **F1** : moyenne +0,002, **28 sur 50**, avec des écarts selon le modèle de −0,002 à +0,005, plus petits que l'écart-type de la différence (0,005 à 0,015).
- **Conclusion de l'étape :** ajouter le cluster aux variables initiales ne produit **aucune amélioration mesurable**, pour aucun des 5 modèles.

### 6.4 Étape 2 : A vs B (le Feature Engineering seul)

In [ ]:
run("B")
report("B", "A")

**Lecture :**
- **Moyenne nulle** : ROC-AUC −0,0003 (25 plis sur 50), F1 −0,0011 (25 sur 50).
- Par modèle, les effets sont petits et de signes différents : le Random Forest et le SVM gagnent environ +0,002 de ROC-AUC (7 et 8 plis sur 10), le Decision Tree perd environ −0,005 (2 plis sur 10).
- **Conclusion de l'étape :** le Feature Engineering n'apporte pas de gain d'ensemble sur ces données ; les petits effets par modèle ne justifient pas de conclure à un vrai apport.

### 6.5 Étape 3 : B vs D (le cluster en plus du Feature Engineering)

In [ ]:
run("D")
report("D", "B")

**Lecture :**
- ROC-AUC +0,0001 (23 plis sur 50) et F1 +0,0006 (23 sur 50) : toujours du niveau du hasard.
- Un détail : pour la Logistic Regression, le F1 baisse de 0,003 avec **0 pli gagné sur 10** et un écart-type faible. L'effet est donc régulier, mais sa taille (0,003) est négligeable.
- **Conclusion de l'étape :** même avec le Feature Engineering déjà présent, le cluster n'apporte rien de mesurable.

### 6.6 Vérification sur le test (A vs C)

Simple contrôle sur des clients jamais vus, pas un critère de choix.

In [ ]:
rows = []
for name in MODEL_NAMES:
    a = test_metrics(build_pipeline(name, "smote", False, False).fit(X_train_c, y_train), X_test_c, y_test)
    c = test_metrics(build_pipeline(name, "smote", True, False).fit(X_train_c, y_train), X_test_c, y_test)
    rows.append({"model": name, "roc_auc A": a["roc_auc"], "roc_auc C": c["roc_auc"],
                 "f1 A": a["f1"], "f1 C": c["f1"], "recall A": a["recall"], "recall C": c["recall"]})
pd.DataFrame(rows).set_index("model").round(3)

**Lecture :** sur le test, le ROC-AUC bouge d'au plus 0,002, et le F1 varie de −0,009 à +0,005 **dans des sens différents selon les modèles** (Logistic Regression et Decision Tree montent un peu, Random Forest, SVM et Gradient Boosting baissent). Pas d'effet net, comme en validation croisée.

### 6.7 Pourquoi le cluster n'aide-t-il pas ?

- **Il ne contient pas d'information nouvelle.** Le cluster est une fonction des mêmes variables que celles que le modèle reçoit déjà (K-Means a été appliqué à ces mêmes colonnes). Le modèle peut retrouver ces combinaisons lui-même, surtout les arbres et le boosting.
- **Il a été construit sans `Churn`.** Un clustering non supervisé ne peut pas ajouter d'information sur la cible qui ne soit déjà dans les variables.
- **Le fait que les segments aient des taux de churn très différents ne contredit rien.** Cet écart s'explique par des variables que le modèle connaît déjà (fibre, contrat mensuel, ancienneté courte).
- **Le même raisonnement vaut pour le Feature Engineering** : `n_services`, `avg_charge` et `tenure_group` sont des combinaisons de variables existantes.

**Précaution sur la méthode :** K-Means a été ajusté sur tout le train, puis utilisé dans la validation croisée : il y a une légère fuite d'information qui favoriserait le cluster. Comme aucun gain n'apparaît malgré cela, la conclusion n'en est que plus solide. Pour une validation parfaite, on placerait K-Means dans le Pipeline.

**À quoi sert le cluster malgré tout ?** À **comprendre et présenter** les clients : les 4 segments donnent une lecture métier (qui sont les clients à risque, quelles actions de rétention cibler) et pourront être affichés dans l'application Streamlit.

**Limite de cette conclusion :** elle vaut pour ces données, ces 5 modèles et ce clustering (K-Means, k = 4). Elle ne dit pas qu'un autre clustering ne pourrait jamais aider.

## 7. Conclusion et suite

**Ce qu'on a appris**
- L'accuracy trompe sur ce jeu déséquilibré (73,5 % sans rien apprendre) : on juge sur Recall, F1 et ROC-AUC.
- Sans traitement, les 5 modèles ne détectent qu'environ la moitié des clients qui partent. `class_weight` et SMOTE relèvent le Recall à ≈ 0,79 (Logistic Regression, Decision Tree, SVM) en échange d'une Precision plus faible : c'est un compromis métier, pas un gain gratuit.
- Pour la Logistic Regression, le rééquilibrage revient presque à abaisser le seuil de décision ; ce n'est pas vrai de tous les modèles (SVM, Random Forest).
- **Ni le cluster ni le Feature Engineering n'améliorent la prédiction de façon mesurable** : l'information qu'ils portent est déjà dans les variables d'origine. Le cluster reste utile pour **interpréter** les clients.
- Logistic Regression et Gradient Boosting restent les deux meilleurs candidats (meilleur ROC-AUC).

**Pas encore fait (prochaines étapes)**
1. **Optimisation** : validation croisée + `GridSearchCV` / `RandomizedSearchCV` sur ces deux modèles, comparaison avant / après, choix du seuil de décision. On pourra y revérifier l'apport du cluster et du Feature Engineering avec des modèles réglés.
2. **MLflow** : suivi des expériences (hyperparamètres, métriques, modèle, artifacts).
3. **Streamlit** (profil/cluster, prédiction, probabilité) et **Docker**.
4. README et livraison.